In [ ]:
# Manuscript Figure 7 / Supp Figure 8 -----------------------------------------
#------------------------------------------------------------------------------

library(Seurat)
#library(SeuratDisk)
library(ArchR)
library(BSgenome.Hsapiens.UCSC.hg38)
library(AUCell)
library(readxl)
library(patchwork)
library(data.table)
library(ggplot2)
library(GSEABase)
library(broom)
library(ggrepel)
library(gridExtra) # Or use the `patchwork` package for flexibility
library(ComplexHeatmap)
library(circlize)
#library(reshape2) # masks mutate from dplyr
library(dplyr)
library(tidyr)
library(ggpubr)


set.seed(1)
addArchRThreads(threads = 42) 
addArchRGenome("hg38")

proj_dir = '/Aging/'

### directories
data_dir = '/singleCell/Ruzicka_snRNA_Seq/data/'
data_dir2 = '/Aging/Data/Anderson/snRNA_Seq/'
out_dir2 = '/Aging/Aging_Clock/Output/'

PLOTDIR <- '/Aging/Aging_Clock/Plots/'
setwd(PLOTDIR)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect



                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ===

In [2]:
# Base metadata
metadata = read.csv('/Aging/Data/Anderson/Multiomics/Raw/GSE214979_cell_metadata.csv')

# Extract metadata from ATAC Project
proj <- loadArchRProject(path = "/Aging/Data/Anderson/Multiomics/ATAC/ArchR/ArchRProject_Neurons", force = FALSE, showLogo = FALSE)
metadata1 <- as.data.frame(getCellColData(proj))
metadata1 <- metadata1[1:15]
metadata1$X <- rownames(metadata1)
metadata1$X = gsub('GSE214979_atac_fragments#','',metadata1$X)

# ER_score
ER_score <- read.csv(paste0(data_dir2,"GSE214979_ER_score.csv"))
ER_score$X = gsub('GSE214979_atac_fragments#','',ER_score$X)

# AUCell
AUCell <- read.csv(paste0(data_dir2,"GSE214979_metadata_AUCell_all_scores.csv"), row.names = 1)
AUCell <- AUCell[c(5,27,60:91)]
AUCell$geneset0_status <- coalesce(AUCell$geneset_pos0.y, AUCell$geneset_neg0.y)
AUCell$geneset24_status <- coalesce(AUCell$geneset_pos24.y, AUCell$geneset_neg24.y)
q1 = round(quantile(AUCell$geneset_dd.x,0.25,na.rm=T),4)
q2 = round(quantile(AUCell$geneset_dd.x,0.75,na.rm=T),4)
AUCell = AUCell %>% mutate(geneset_dd_status = case_when(geneset_dd.x <= q1 ~ "low",
                                                 geneset_dd.x >= q2 ~ "high"))
#----------------------------------------------------------------

metadata <- left_join(metadata, metadata1, by='X')
metadata <- left_join(AUCell, metadata, by='X')
metadata <- left_join(metadata, ER_score, by='X') # 56215 x 84

# Remove Ex-NRGN from Analysis
metadata <- metadata[-which(metadata$predicted.id.x=='Ex-NRGN'),]

ERROR: Error in loadArchRProject(path = "/home/gabdelha/ArchR/Anderson_Multiomics/ArchRProject_Neurons", : could not find function "loadArchRProject"


**ATAC Metrics**

In [33]:
# Define parameters

#predicted.id.y for class
#predicted.id.x for subclass 

geneset_statuses <- c('TSSEnrichment','PromoterRatio','NucleosomeRatio','Epigenetic_Erosion_Score','nFrags',
                      'geneset_sc0','geneset_sc24','geneset_dd.x')

cell_types <- c("In-PV_Basket","In-PV_Chandelier","In-Reelin","In-Rosehip_CHST9","In-Rosehip_TRPC3","In-SST","In-VIP",
                       "Ex-L2","Ex-L3","Ex-L4_MYLK","Ex-L5b_HTR2C","Ex-L5b_VEN","Ex-L6_CC_SEMA3A","Ex-L6b_SEMA3D","Ex-L6b_SEMA3E","Ex-L23","Ex-L45_LRRK1","Ex-L45_MET","Ex-L56","Ex-L56_CC_NTNG2")


pdf(file = "BoxPlot_Anderson_ATAC_ER_class_reads.pdf", width = 4, height = 5) # 14*7
# Loop through each geneset status
for (i in seq_along(geneset_statuses)) {
  geneset_status <- geneset_statuses[i]
  print(geneset_status)
  metadata1 <- metadata %>% filter(!is.na(.data[[geneset_status]]))
    
  cell_order <- metadata1 %>%
    filter(Status == "Ctrl") %>%
    group_by(predicted.id.y) %>%
    summarise(med = median(.data[[geneset_status]], na.rm = TRUE), .groups = "drop") %>%
    arrange(med) %>%
    pull(predicted.id.y)
  
   metadata1 <- metadata1 %>%
    mutate(predicted.id.y = factor(predicted.id.y, levels = cell_order))
  
  p1 <- ggplot(metadata1, aes(x = .data[['predicted.id.y']], 
                              y = .data[[geneset_status]],
                              fill = Status)) +
    geom_boxplot(position = position_dodge(0.8), color = "#000435", outlier.color = "#000435", alpha = 0.7) +
    scale_fill_manual(values = c("AD" = "#21FE80", "Ctrl" = "#000435")) +
    stat_compare_means(aes(group = Status), method = "wilcox", label = "p.format", angle = 45) +
    theme_minimal() +
    labs(y = geneset_status) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

  print(p1)
}

dev.off()

[1] "TSSEnrichment"
[1] "PromoterRatio"
[1] "NucleosomeRatio"
[1] "Epigenetic_Erosion_Score"
[1] "nFrags"
[1] "geneset_sc0"
[1] "geneset_sc24"
[1] "geneset_dd.x"


pdf 
  2

In [29]:
pdf(file = "BoxPlot_Anderson_ATAC_ER_class_QC.pdf", width = 6, height = 6)

geneset_statuses <- c('TSSEnrichment','PromoterRatio','NucleosomeRatio','nFrags')
# Reshape to long format

metadata$predicted.id.y <- factor(metadata$predicted.id.y, levels = c('Inhibitory', 'Excitatory'))
metadata_long <- metadata %>%
  dplyr::select(predicted.id.y, Status, all_of(geneset_statuses)) %>%
  pivot_longer(cols = all_of(geneset_statuses), 
               names_to = "Geneset", 
               values_to = "Value") %>%
  filter(!is.na(Value))

# Create 2x2 grid plot
p_combined <- ggplot(metadata_long, aes(x = predicted.id.y, y = Value, fill = Status)) +
  geom_boxplot(position = position_dodge(0.8), color = "#000435", outlier.color = "#000435", alpha = 0.7) +
  scale_fill_manual(values = c("AD" = "#21FE80", "Ctrl" = "#000435")) +
  stat_compare_means(aes(group = Status), method = "wilcox", label = "p.format") +
  theme_minimal() +
  facet_wrap(~ Geneset, scales = "free_y", ncol = 2) +
  labs(x = "Cell Type", y = "Score") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

print(p_combined)

dev.off()

pdf 
  2

In [25]:
pdf(file = "BoxPlot_Anderson_ATAC_ER_genesets_QC_Inh.pdf", width = 5.5, height = 6)

# Define parameters
geneset_statuses <- c("geneset_dd_status", "geneset24_status", "geneset0_status")
geneset_labels <- c("DNA Damage", 'Intrinsic Age', 'Systemic Age')

Inh <- metadata[metadata$predicted.id.y %in% c("Inhibitory"),]
Exc <- metadata[metadata$predicted.id.y %in% c("Excitatory"),]

# Loop through each geneset status
for (i in seq_along(geneset_statuses)) {
  geneset_status <- geneset_statuses[i]
  geneset_label <- geneset_labels[i]
  print(geneset_status)
  print(geneset_label)

  metadata1 <- filter(Inh, !is.na(.data[[geneset_status]]))
    
    # Split by Status
    # Positive vs Negative 
    # High vs low    
  p1 <- ggplot(metadata1, aes_string(x = geneset_status, y = "nFrags")) +
    geom_boxplot(fill= c("#21FE80","#21FE80","#000435","#000435"), color = "#000435", outlier.color = "#000435", alpha = 0.7) +
    facet_wrap(~Status, ncol = 2) +
    stat_compare_means(method = "wilcox", label = "p.format") +
    theme_minimal() +
    labs(x = geneset_label, y = "Number of Fragments") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

  p2 <- ggplot(metadata1, aes_string(x = geneset_status, y = "NucleosomeRatio")) +
    geom_boxplot(fill= c("#21FE80","#21FE80","#000435","#000435"),color = "#000435", outlier.color = "#000435", alpha = 0.7) +
    facet_wrap(~Status, ncol = 2) +
    stat_compare_means(method = "wilcox", label = "p.format") +
    theme_minimal() +
    labs(x = geneset_label, y = "Nucleosome Ratio") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
  p3 <- ggplot(metadata1, aes_string(x = geneset_status, y = "PromoterRatio")) +
    geom_boxplot(fill= c("#21FE80","#21FE80","#000435","#000435"), color = "#000435", outlier.color = "#000435", alpha = 0.7) +
    facet_wrap(~Status, ncol = 2) +
    stat_compare_means(method = "wilcox", label = "p.format") +
    theme_minimal() +
    labs(x = geneset_label, y = "Promoter Ratio") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
  p4 <- ggplot(metadata1, aes_string(x = geneset_status, y = "TSSEnrichment")) +
    facet_wrap(~Status, ncol = 2) +
    geom_boxplot(fill= c("#21FE80","#21FE80","#000435","#000435"),color = "#000435", outlier.color = "#000435", alpha = 0.7) +
    stat_compare_means(method = "wilcox", label = "p.format") +
    theme_minimal() +
    labs(x = geneset_label, y = "TSS Enrichment") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

  # Combine the plots in a 2x2 grid
  combined_plot <- (p1 + p2) / (p3 + p4)
  # Print the combined plot
  print(combined_plot)

}

dev.off()

[1] "geneset_dd_status"
[1] "DNA Damage"


Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 674 rows containing non-finite outside the scale range
(`stat_compare_means()`).”


[1] "geneset24_status"
[1] "Intrinsic Age"


Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 470 rows containing non-finite outside the scale range
(`stat_compare_means()`).”


[1] "geneset0_status"
[1] "Systemic Age"


Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_compare_means()`).”
Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 237 rows containing non-finite outside the scale range
(`stat_compare_means()`).”


pdf 
  2

In [26]:
# Define parameters
geneset_statuses <- c("geneset0_status","geneset24_status","geneset_dd_status")
geneset_labels <- c('Systemic Aging','Intrinsic Aging',"DNA Damage")

Inh <- metadata[metadata$predicted.id.y %in% c("Inhibitory"),]
Exc <- metadata[metadata$predicted.id.y %in% c("Excitatory"),]

pdf(file = "BoxPlot_Anderson_ATAC_ER_geneset_Inh_reads.pdf", width = 2.5, height = 5)

# Loop through each geneset status
for (i in seq_along(geneset_statuses)) {
  geneset_status <- geneset_statuses[i]
  geneset_label <- geneset_labels[i]
  print(geneset_status)
  print(geneset_label)
  metadata1 <- filter(Inh, !is.na(.data[[geneset_status]]))

    # Split by Status
    # Positive vs Negative 
    # High vs low    
  p1 <- ggplot(metadata1, aes_string(x = geneset_status, y = "Epigenetic_Erosion_Score")) +
    geom_boxplot(fill = c("#21FE80","#21FE80","#000435","#000435"), color = "#000435", outlier.color = "#000435", alpha = 0.7) +
    facet_wrap(~Status, ncol = 2) +
    stat_compare_means(method = "wilcox", label = "p.format") +
    theme_minimal() +
    labs(x = geneset_label, y = "Epigenetic Erosion Score") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

   print(p1)

    }
dev.off()

[1] "geneset0_status"
[1] "Systemic Aging"


Warning message:
“Removed 238 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 238 rows containing non-finite outside the scale range
(`stat_compare_means()`).”


[1] "geneset24_status"
[1] "Intrinsic Aging"


Warning message:
“Removed 471 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 471 rows containing non-finite outside the scale range
(`stat_compare_means()`).”


[1] "geneset_dd_status"
[1] "DNA Damage"


Warning message:
“Removed 675 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 675 rows containing non-finite outside the scale range
(`stat_compare_means()`).”


pdf 
  2

In [23]:
# Correlation 

pdf(file = "Heatmap_Anderson_ATAC_QC.pdf", width = 6, height = 6)
geneset_statuses <- c('TSSEnrichment','PromoterRatio','NucleosomeRatio','Epigenetic_Erosion_Score')
#geneset_statuses <- c('geneset_sc0','geneset_sc24','geneset_dd.x','Epigenetic_Erosion_Score')
tbl <- na.omit(metadata[, geneset_statuses])
cor_matrix <- cor(tbl)
h1 <- Heatmap(cor_matrix, col = colorRamp2(c(-1, 0, 1), c("#000435", "#FFFFFF", "#21FE80")),
              cluster_rows = T, cluster_columns = T,
              cell_fun = function(j, i, x, y, w, h, fill){
                  # Draw the border for each cell
                  grid.rect(x, y, w, h, gp = gpar(col = "white", lwd = 2, fill = NA))
                  grid.text(round(cor_matrix[i, j], 2), x, y, gp = gpar(fontsize = 15, col = "#000435"))
              })
draw(h1)
dev.off()

pdf 
  2

In [ ]:
# Epigenomic Erosion Distribution Shift #

In [52]:
# Distribution 

# Classes 
pdf(file = "Density_Anderson_ATAC_ER_class.pdf", width = 10, height = 6)
par(mfrow = c(1,2))

cell_types <- sort(unique(metadata$predicted.id.y))
for (ctype in cell_types) {
  subset_data <- metadata[metadata$predicted.id.y == ctype, ]

  AD <- subset_data[subset_data$Status == "AD", ]
  Ctrl <- subset_data[subset_data$Status == "Ctrl", ]

  ks_res <- ks.test(
  na.omit(AD$Epigenetic_Erosion_Score),
  na.omit(Ctrl$Epigenetic_Erosion_Score))

  plot(density(na.omit(AD$Epigenetic_Erosion_Score), adjust = 1), 
       col = '#21FE80', lwd = 2, xlim = c(-0.2, 0.2), ylim = c(0, 12),
       main = paste(ctype), xlab = "Epigenetic Erosion Score", ylab = "Density")
  lines(density(na.omit(Ctrl$Epigenetic_Erosion_Score), adjust = 1), col = '#000435', lwd = 2)

  abline(v = 0, col = 'darkred', lwd = 2, lty = 2)
  abline(v = mean(AD$Epigenetic_Erosion_Score), col = '#21FE80', lwd = 2, lty = 2)  
  abline(v = mean(Ctrl$Epigenetic_Erosion_Score), col = '#000435', lwd = 2, lty = 2)

  legend("topright", legend = c("AD", "Control"), col = c('#21FE80', '#000435'), lwd = 2, bty = "n")
  text(x = -0.19, y = 11.5, labels = paste0("KS D = ", round(ks_res$statistic, 2), "\n", "p-value = ", round(ks_res$p.value, 4)),adj = c(0, 1),cex = 0.9)
}

dev.off()


# Subclasses
pdf(file = "Density_Anderson_ATAC_ER_subclass.pdf", width = 15, height = 10)
par(mfrow = c(4,5))

#cell_types <- sort(unique(metadata$predicted.id.x))
cell_types <- c("In-PV_Basket","In-PV_Chandelier","In-Reelin","In-Rosehip_CHST9","In-Rosehip_TRPC3","In-SST","In-VIP",
                       "Ex-L2","Ex-L3","Ex-L4_MYLK","Ex-L5b_HTR2C","Ex-L5b_VEN","Ex-L6_CC_SEMA3A","Ex-L6b_SEMA3D","Ex-L6b_SEMA3E","Ex-L23","Ex-L45_LRRK1","Ex-L45_MET","Ex-L56","Ex-L56_CC_NTNG2")

for (ctype in cell_types) {
  subset_data <- metadata[metadata$predicted.id.x == ctype, ]

  AD <- subset_data[subset_data$Status == "AD", ]
  Ctrl <- subset_data[subset_data$Status == "Ctrl", ]

  ks_res <- ks.test(
  na.omit(AD$Epigenetic_Erosion_Score),
  na.omit(Ctrl$Epigenetic_Erosion_Score))

  plot(density(na.omit(AD$Epigenetic_Erosion_Score), adjust = 1), 
       col = '#21FE80', lwd = 2, xlim = c(-0.2, 0.2), ylim = c(0, 12),
       main = paste(ctype), xlab = "Epigenetic Erosion Score", ylab = "Density")
  lines(density(na.omit(Ctrl$Epigenetic_Erosion_Score), adjust = 1), col = '#000435', lwd = 2)

  abline(v = 0, col = 'darkred', lwd = 2, lty = 2)
  abline(v = mean(AD$Epigenetic_Erosion_Score), col = '#21FE80', lwd = 2, lty = 2)  
  abline(v = mean(Ctrl$Epigenetic_Erosion_Score), col = '#000435', lwd = 2, lty = 2)

  legend("topright", legend = c("AD", "Control"), col = c('#21FE80', '#000435'), lwd = 2, bty = "n")
  text(x = -0.19, y = 11.5, labels = paste0("KS D = ", round(ks_res$statistic, 2), "\n", "p-value = ", round(ks_res$p.value, 4)),adj = c(0, 1),cex = 0.9)
}

dev.off()

pdf 
  2

pdf 
  2